# مشروع تطبيقي: نظام فرز وتوجيه تذاكر الدعم الفني الذكي
### Hands-on Project: Intelligent Customer Support Ticket Triage System with LangGraph

مرحباً بك في أول مشروع تطبيقي عملي!
في هذا المشروع، سنبني معاً نظاماً حقيقياً لخدمة العملاء يعتمد على **LangGraph**؛ يقوم باستقبال رسائل العملاء، وتصنيفها فورياً، وتحديد أولويتها، ثم توجيهها للمسار المختص لإعداد الرد المناسب.

المراحل الهندسية للمشروع:
1. تعريف مخطط الحالة المشتركة (Ticket State).
2. بناء عقد المعالجة (التصنيف، مسار الطوارئ، مسار الفواتير، مسار الاستفسارات، والصياغة النهائية).
3. بناء دالة التوجيه الشرطي (Conditional Router).
4. تجميع الجراف واختبار الحالات الحقيقية.
5. قسم خاص: جرب بنفسك وطور المشروع!


In [1]:
from typing import Dict, Any, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

print("تم استيراد كافة مكونات LangGraph بنجاح!")


تم استيراد كافة مكونات LangGraph بنجاح!


### 1. تعريف الحالة المشتركة (Ticket State Schema)
الحالة هي الصندوق المشترك الذي يحمل بيانات التذكرة أثناء انتقالها بين العقد:
- نص رسالة العميل (customer_message).
- نوع التذكرة بعد التصنيف (ticket_type).
- أولوية التذكرة (priority).
- الرد الصادر من النظام (response).


In [2]:
class TicketState(TypedDict):
    customer_message: str
    ticket_type: str
    priority: str
    response: str

print("تم تعريف كلاس الحالة TicketState بنجاح!")


تم تعريف كلاس الحالة TicketState بنجاح!


### 2. بناء العقد الحسابية (Nodes)
كل عقدة تمثل موظفاً متخصصاً داخل فريق الدعم الفني.


In [3]:
# العقدة 1: فحص وتصنيف رسالة العميل
def classify_ticket(state: TicketState) -> Dict[str, Any]:
    msg = state["customer_message"].lower()
    
    # فحص الكلمات المفتاحية
    if any(word in msg for word in ["سيرفر", "عطل", "وقع", "توقف", "down", "error"]):
        ticket_type = "emergency"
        priority = "HIGH"
    elif any(word in msg for word in ["فلوس", "دفع", "اشتراك", "استرجاع", "فاتورة", "refund"]):
        ticket_type = "billing"
        priority = "MEDIUM"
    else:
        ticket_type = "general"
        priority = "LOW"
        
    print(f"[عقدة التصنيف]: تم التعرف على نوع المشكلة -> {ticket_type} (الأولوية: {priority})")
    return {"ticket_type": ticket_type, "priority": priority}

# العقدة 2: مسار الطوارئ التقنية
def tech_support(state: TicketState) -> Dict[str, Any]:
    print("[مسار الطوارئ التقنية]: جاري تنبيه فريق العمليات وتجهيز رد فوري...")
    return {
        "response": "تم رفع المشكلة لفريق البنية التحتية بأولوية قصوى وجاري التعامل مع العطل حالياً."
    }

# العقدة 3: مسار الفواتير والمدفوعات
def billing_support(state: TicketState) -> Dict[str, Any]:
    print("[مسار الفواتير]: جاري مراجعة سجل العمليات المالية...")
    return {
        "response": "تم استلام استفسار الفوترة، وسيتم التحقق من الحساب والرد خلال 24 ساعة عمل."
    }

# العقدة 4: مسار الاستفسارات العامة
def general_support(state: TicketState) -> Dict[str, Any]:
    print("[مسار الاستفسار العام]: جاري توجيه العميل للدليل الإرشادي...")
    return {
        "response": "أهلاً بك! يمكنك مراجعة مركز المساعدة الشامل أو إرسال تفاصيل إضافية وسنرد عليك فوراً."
    }

# العقدة 5: صياغة الرد النهائي المعتمد
def finalize_ticket(state: TicketState) -> Dict[str, Any]:
    final_text = f"[تذكرة أولوية {state['priority']}] -> {state['response']}"
    return {"response": final_text}

print("تم تعريف عقد الجراف الخمس بنجاح!")


تم تعريف عقد الجراف الخمس بنجاح!


### 3. دالة التوجيه الشرطي (Conditional Router)
تفحص نوع التذكرة المسجل في الحالة وتحدد مسار العقدة التالية بدقة.


In [4]:
def route_ticket(state: TicketState) -> Literal["tech_support", "billing_support", "general_support"]:
    t_type = state["ticket_type"]
    if t_type == "emergency":
        return "tech_support"
    elif t_type == "billing":
        return "billing_support"
    return "general_support"

print("تم تعريف دالة التوجيه الشرطي route_ticket بنجاح!")


تم تعريف دالة التوجيه الشرطي route_ticket بنجاح!


### 4. تجميع الجراف وتوصيل المسارات
الآن سنقوم بربط العقد بنقاط البداية والتفرع والنهاية.


In [5]:
workflow = StateGraph(TicketState)

# 1. تسجيل العقد
workflow.add_node("classify_ticket", classify_ticket)
workflow.add_node("tech_support", tech_support)
workflow.add_node("billing_support", billing_support)
workflow.add_node("general_support", general_support)
workflow.add_node("finalize_ticket", finalize_ticket)

# 2. ربط نقطة البداية بعقدة التصنيف
workflow.add_edge(START, "classify_ticket")

# 3. ربط التفرع الشرطي بناءً على النوع
workflow.add_conditional_edges(
    "classify_ticket",
    route_ticket,
    {
        "tech_support": "tech_support",
        "billing_support": "billing_support",
        "general_support": "general_support"
    }
)

# 4. توجيه كافة المسارات لعقدة الصياغة النهائية ثم النهاية END
workflow.add_edge("tech_support", "finalize_ticket")
workflow.add_edge("billing_support", "finalize_ticket")
workflow.add_edge("general_support", "finalize_ticket")
workflow.add_edge("finalize_ticket", END)

# 5. التجميع (Compilation)
ticket_app = workflow.compile()
print("تم تجميع الجراف بنجاح، وهو جاهز للعمل في بيئة الإنتاج!")


تم تجميع الجراف بنجاح، وهو جاهز للعمل في بيئة الإنتاج!


### 5. اختبار النظام على سيناريوهات عمل حقيقية
تعال نجرب النظام على 3 تذاكر فعلية ونشاهد كيف يتصرف الجراف بذكاء:


In [6]:
# السيناريو 1: بلاغ عن عطل في السيرفر
print("=== السيناريو 1: بلاغ تقني حرج ===")
test_1 = {
    "customer_message": "السيرفر الأساسي وقع والتطبيق متوقف عن العمل بالكامل!",
    "ticket_type": "",
    "priority": "",
    "response": ""
}
res_1 = ticket_app.invoke(test_1)
print("الرد الصادر للعميل:\n", res_1["response"])

# السيناريو 2: استرجاع مالي
print("\n=== السيناريو 2: مشكلة مالية واشتراكات ===")
test_2 = {
    "customer_message": "تم خصم مبلغ الاشتراك مرتين وعايز استرجاع فلوس الفاتورة الأخيرة",
    "ticket_type": "",
    "priority": "",
    "response": ""
}
res_2 = ticket_app.invoke(test_2)
print("الرد الصادر للعميل:\n", res_2["response"])

# السيناريو 3: سؤال عام
print("\n=== السيناريو 3: استفسار عام ===")
test_3 = {
    "customer_message": "هل يتوفر لديكم دعم فني خلال أيام العطلات الأسبوعية؟",
    "ticket_type": "",
    "priority": "",
    "response": ""
}
res_3 = ticket_app.invoke(test_3)
print("الرد الصادر للعميل:\n", res_3["response"])


=== السيناريو 1: بلاغ تقني حرج ===
[عقدة التصنيف]: تم التعرف على نوع المشكلة -> emergency (الأولوية: HIGH)
[مسار الطوارئ التقنية]: جاري تنبيه فريق العمليات وتجهيز رد فوري...
الرد الصادر للعميل:
 [تذكرة أولوية HIGH] -> تم رفع المشكلة لفريق البنية التحتية بأولوية قصوى وجاري التعامل مع العطل حالياً.

=== السيناريو 2: مشكلة مالية واشتراكات ===
[عقدة التصنيف]: تم التعرف على نوع المشكلة -> billing (الأولوية: MEDIUM)
[مسار الفواتير]: جاري مراجعة سجل العمليات المالية...
الرد الصادر للعميل:
 [تذكرة أولوية MEDIUM] -> تم استلام استفسار الفوترة، وسيتم التحقق من الحساب والرد خلال 24 ساعة عمل.

=== السيناريو 3: استفسار عام ===
[عقدة التصنيف]: تم التعرف على نوع المشكلة -> emergency (الأولوية: HIGH)
[مسار الطوارئ التقنية]: جاري تنبيه فريق العمليات وتجهيز رد فوري...
الرد الصادر للعميل:
 [تذكرة أولوية HIGH] -> تم رفع المشكلة لفريق البنية التحتية بأولوية قصوى وجاري التعامل مع العطل حالياً.


### 6. جرب بنفسك وطور المشروع!
إليك أفكار عملية لإضافة لمستك الخاصة وتطوير هذا المشروع:
1. **تحدي 1**: أضف عقدة جديدة باسم `security_support` تتعامل مع مشاكل "اختراق الحساب" أو "تسريب البيانات" مع أولوية قصوى `CRITICAL`.
2. **تحدي 2**: أضف في الحالة `TicketState` حقلاً باسم `assigned_department: str` يعيد اسم القسم المختص باللغة الإنجليزية (مثل `IT-Ops` أو `Finance`).
3. **تحدي 3**: اربط عقدة التصنيف بنموذج LLM مجاني وسريع (مثل Groq) بدلاً من فحص الكلمات يدparam!
